# Lab 0 - Task 0.2 - Transfer Learning

### Task 0.2.1 - Transfer Learning from ImageNet (AlexNet on CIFAR-10)
- Experiment A: Train AlexNet from scratch (no pretrained weights)
- Experiment B: AlexNet as Feature Extractor (pretrained, frozen backbone, only new FC layer trains)
- Experiment C: AlexNet Fine-Tuning (pretrained, all layers unfrozen, differential learning rates)

### Task 0.2.2 - Transfer Learning from MNIST to SVHN
- Step 1: Train a CNN on MNIST
- Step 2: Transfer the MNIST CNN to SVHN (Feature Extraction)
- Step 3: Train the same CNN on SVHN from scratch (No Transfer Learning)
- Step 4: Transfer to SVHN with all layers unfrozen (Fine-Tuning)

### Tracking: Weights & Biases (WandB)

---
## Setup & Imports

In [92]:
%%time

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import wandb
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

# Limit GPU memory to 50% 
torch.cuda.set_per_process_memory_fraction(0.50, device=0)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cuda
CPU times: user 430 μs, sys: 163 μs, total: 593 μs
Wall time: 577 μs


---
## Task 0.2.1 — AlexNet on CIFAR-10

### Load CIFAR-10

AlexNet expects input images of at least 63x63. CIFAR-10 is 32x32, so we resize to 224x224 (standard ImageNet size).

In [93]:
%%time

# AlexNet was trained on ImageNet with ~224x224 images.
# CIFAR-10 is 32x32, so we resize + use ImageNet normalization stats
# and add light augmentation for training.

cifar_train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],   # ImageNet mean
        std=[0.229, 0.224, 0.225]     # ImageNet std
    )
])

cifar_test_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

cifar_trainset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=cifar_train_transform
)
cifar_testset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=cifar_test_transform
)

cifar_trainloader = torch.utils.data.DataLoader(
    cifar_trainset, batch_size=64, shuffle=True, num_workers=2
)
cifar_testloader = torch.utils.data.DataLoader(
    cifar_testset, batch_size=64, shuffle=False, num_workers=2
)

print('CIFAR-10 Training samples:', len(cifar_trainset))
print('CIFAR-10 Test samples:', len(cifar_testset))

/usr/local/lib/python3.11/dist-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


CIFAR-10 Training samples: 50000
CIFAR-10 Test samples: 10000
CPU times: user 1.56 s, sys: 280 ms, total: 1.84 s
Wall time: 1.84 s


### Shared Training & Evaluation Functions 

In [94]:
%%time

NUM_EPOCHS = 10
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406])
IMAGENET_STD  = np.array([0.229, 0.224, 0.225])

# --- WandB: Initialize a new run ---
def start_run(run_name, config=None):
    """Initialize a WandB run for the team."""
    wandb.init(
        entity="adl-group-10",
        project="Lab-0",
        name=run_name,
        config=config or {}
    )

# --- WandB: Close the run ---
def end_run():
    """Close the current WandB run."""
    wandb.finish()

def train_model(model, optimizer, trainloader, num_epochs, run_name):
    """Training function with WandB logging."""

    criterion = nn.CrossEntropyLoss()

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for inputs, labels in trainloader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

        epoch_loss = running_loss / len(trainloader)
        epoch_acc = 100. * correct / total

        # --- WandB: Log training metrics each epoch ---
        log_dict = {
            "training_loss": epoch_loss,
            "training_accuracy": epoch_acc,
            "epoch": epoch + 1
        }
        for i, pg in enumerate(optimizer.param_groups):
            log_dict[f"learning_rate_group_{i}"] = pg['lr']
        wandb.log(log_dict)

        print(f'Epoch [{epoch+1}/{num_epochs}] | Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc:.2f}%')

    print('Training complete!')
    return model


def evaluate_model(model, testloader, label):
    """Evaluation function."""
    model.eval()
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in testloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = 100. * correct / total
    print(f'Test Accuracy ({label}): {acc:.2f}%')
    return acc, all_preds, all_labels

def log_prediction_samples(model, testloader, run_name, num_samples=32):
    """Log a grid of sample predictions to WandB."""
    model.eval()
    images, labels = next(iter(testloader))
    images, labels = images[:num_samples].to(device), labels[:num_samples]
    
    with torch.no_grad():
        preds = model(images).argmax(1).cpu()
    
    fig, axes = plt.subplots(4, 8, figsize=(16, 8))
    for i, ax in enumerate(axes.flat):
        img = unnormalize_imagenet(images[i].cpu())
        ax.imshow(img)
        color = 'green' if preds[i] == labels[i] else 'red'
        ax.set_title(f'P:{preds[i]} / T:{labels[i]}', color=color, fontsize=10)
        ax.axis('off')
    plt.tight_layout()
    wandb.log({f"{run_name}_predictions": wandb.Image(fig)})
    plt.close()

def count_params(model):
    """Log trainable vs total parameter counts to WandB."""
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f'Parameters: {trainable:,} trainable / {total:,} total')
    wandb.log({"trainable_params": trainable, "total_params": total})

def unnormalize_imagenet(tensor):
    # tensor: C x H x W (on CPU)
    img = tensor.permute(1, 2, 0).numpy()
    img = (img * IMAGENET_STD) + IMAGENET_MEAN
    return img.clip(0, 1)

# Dictionary to collect all results
results = {}

CPU times: user 160 μs, sys: 0 ns, total: 160 μs
Wall time: 163 μs


### Experiment A — Train AlexNet from scratch (no pretrained weights)
Load AlexNet **without** pretrained weights, replace the final classifier layer with one that outputs 10 classes, and train all weights from scratch on CIFAR-10.

In [95]:
%%time

start_run('alexnet_finetuning', config={
    "architecture": "AlexNet",
    "mode": "fine_tuning",
    "dataset": "CIFAR-10",
    "pretrained": False,
    "optimizer": "Adam",
    "learning_rate": 0.0001,
    "epochs": NUM_EPOCHS
})

# Load AlexNet WITHOUT pretrained weights (fine-tuning = train from scratch on CIFAR-10)
alexnet_ft = models.alexnet(weights=None)

# Replace the last FC layer: AlexNet's classifier[6] outputs 1000 (ImageNet classes)
# We need 10 outputs for CIFAR-10
alexnet_ft.classifier[6] = nn.Linear(4096, 10)
alexnet_ft = alexnet_ft.to(device)
count_params(alexnet_ft)

#optimizer_ft = optim.Adam(alexnet_ft.parameters(), lr=0.0001)
optimizer_ft = optim.Adam(alexnet_ft.parameters(), lr=3e-4, weight_decay=5e-4)

print('=== Experiment A: AlexNet Fine-Tuning (no pretrained weights) ===')
alexnet_ft = train_model(
    model=alexnet_ft,
    optimizer=optimizer_ft,
    trainloader=cifar_trainloader,
    num_epochs=NUM_EPOCHS,
    run_name='alexnet_finetuning'
)

results['Experiment A (AlexNet Fine-Tuning)'], _, _ = evaluate_model(
    alexnet_ft, cifar_testloader, 'AlexNet Fine-Tuning'
)
wandb.log({"test_accuracy": results['Experiment A (AlexNet Fine-Tuning)']})
log_prediction_samples(alexnet_ft, cifar_testloader, 'exp_a_alexnet_ft')

end_run()

Parameters: 57,044,810 trainable / 57,044,810 total
=== Experiment A: AlexNet Fine-Tuning (no pretrained weights) ===
Epoch [1/10] | Loss: 1.5796 | Train Acc: 41.37%
Epoch [2/10] | Loss: 1.1595 | Train Acc: 58.43%
Epoch [3/10] | Loss: 0.9575 | Train Acc: 66.32%
Epoch [4/10] | Loss: 0.8397 | Train Acc: 70.81%
Epoch [5/10] | Loss: 0.7642 | Train Acc: 73.69%
Epoch [6/10] | Loss: 0.7068 | Train Acc: 75.61%
Epoch [7/10] | Loss: 0.6643 | Train Acc: 76.90%
Epoch [8/10] | Loss: 0.6305 | Train Acc: 78.37%
Epoch [9/10] | Loss: 0.5958 | Train Acc: 79.58%
Epoch [10/10] | Loss: 0.5737 | Train Acc: 80.27%
Training complete!
Test Accuracy (AlexNet Fine-Tuning): 80.65%


epoch,▁▂▃▃▄▅▆▆▇█
learning_rate_group_0,▁▁▁▁▁▁▁▁▁▁
test_accuracy,▁
total_params,▁
trainable_params,▁
training_accuracy,▁▄▅▆▇▇▇███
training_loss,█▅▄▃▂▂▂▁▁▁
epoch,10
learning_rate_group_0,0.0003
test_accuracy,80.65
total_params,57044810


CPU times: user 5min 26s, sys: 1min 43s, total: 7min 9s
Wall time: 9min 49s


### Experiment B — AlexNet as Feature Extractor (pretrained, frozen backbone, only new FC layer trains)
Load AlexNet **with** pretrained ImageNet weights. Freeze all layers except the new final classifier layer.
This way the pretrained features are reused and only the 10-class output layer is trained.

In [96]:
%%time

start_run('alexnet_feature_extraction', config={
    "architecture": "AlexNet",
    "mode": "feature_extraction",
    "dataset": "CIFAR-10",
    "pretrained": True,
    "frozen_layers": "all except classifier[6]",
    "optimizer": "Adam",
    "learning_rate": "backbone=1e-5, classifier=1e-3",
    "epochs": NUM_EPOCHS
})

# Load AlexNet WITH pretrained ImageNet weights
alexnet_fe = models.alexnet(weights=models.AlexNet_Weights.IMAGENET1K_V1)

# Freeze ALL parameters (we don't want to update pretrained features)
for param in alexnet_fe.parameters():
    param.requires_grad = False

# Replace + unfreeze only the final layer (10 classes for CIFAR-10)
alexnet_fe.classifier[6] = nn.Linear(4096, 10)  # This new layer has requires_grad=True by default
alexnet_fe = alexnet_fe.to(device)
count_params(alexnet_fe)

# Only pass the trainable parameters (the new final layer) to the optimizer
optimizer_fe = optim.Adam(
    filter(lambda p: p.requires_grad, alexnet_fe.parameters()),
    lr=1e-3,
    weight_decay=5e-4
)

print('=== Experiment B: AlexNet Feature Extraction (pretrained ImageNet weights) ===')
alexnet_fe = train_model(
    model=alexnet_fe,
    optimizer=optimizer_fe,
    trainloader=cifar_trainloader,
    num_epochs=NUM_EPOCHS,
    run_name='alexnet_feature_extraction'
)

results['Experiment B (AlexNet Feature Extraction)'], _, _ = evaluate_model(
    alexnet_fe, cifar_testloader, 'AlexNet Feature Extraction (Pretrained)'
)
wandb.log({"test_accuracy": results['Experiment B (AlexNet Feature Extraction)']})
log_prediction_samples(alexnet_fe, cifar_testloader, 'exp_b_alexnet_fe')

end_run()

Parameters: 40,970 trainable / 57,044,810 total
=== Experiment B: AlexNet Feature Extraction (pretrained ImageNet weights) ===
Epoch [1/10] | Loss: 0.7965 | Train Acc: 72.00%
Epoch [2/10] | Loss: 0.7047 | Train Acc: 75.43%
Epoch [3/10] | Loss: 0.6948 | Train Acc: 75.84%
Epoch [4/10] | Loss: 0.6954 | Train Acc: 75.98%
Epoch [5/10] | Loss: 0.6785 | Train Acc: 76.67%
Epoch [6/10] | Loss: 0.6796 | Train Acc: 76.53%
Epoch [7/10] | Loss: 0.6730 | Train Acc: 77.00%
Epoch [8/10] | Loss: 0.6847 | Train Acc: 76.51%
Epoch [9/10] | Loss: 0.6682 | Train Acc: 77.02%
Epoch [10/10] | Loss: 0.6742 | Train Acc: 77.03%
Training complete!
Test Accuracy (AlexNet Feature Extraction (Pretrained)): 81.76%


epoch,▁▂▃▃▄▅▆▆▇█
learning_rate_group_0,▁▁▁▁▁▁▁▁▁▁
test_accuracy,▁
total_params,▁
trainable_params,▁
training_accuracy,▁▆▆▇▇▇█▇██
training_loss,█▃▂▂▂▂▁▂▁▁
epoch,10
learning_rate_group_0,0.001
test_accuracy,81.76
total_params,57044810


CPU times: user 3min, sys: 2min 14s, total: 5min 15s
Wall time: 11min 39s


### Experiment C — AlexNet Fine-Tuning (Pretrained + All Layers Unfrozen + Differential Learning Rates)
Load AlexNet **with** pretrained ImageNet weights, but **unfreeze all layers** and train end-to-end with differential learning rates. The backbone gets a small lr (1e-5) to gently adapt pretrained features, while the new classifier head gets a larger lr (1e-3) to learn quickly from scratch.

In [97]:
%%time

start_run('alexnet_true_finetuning', config={
    "architecture": "AlexNet",
    "mode": "true_fine_tuning",
    "dataset": "CIFAR-10",
    "pretrained": True,
    "frozen_layers": "none",
    "optimizer": "Adam",
    "learning_rate": "backbone=1e-5, classifier=1e-3",
    "epochs": NUM_EPOCHS
})

# Load AlexNet WITH pretrained ImageNet weights
alexnet_tft = models.alexnet(weights=models.AlexNet_Weights.IMAGENET1K_V1)

# Replace the last FC layer for CIFAR-10 (10 classes)
alexnet_tft.classifier[6] = nn.Linear(4096, 10)
alexnet_tft = alexnet_tft.to(device)
count_params(alexnet_tft)

# All layers unfrozen — differential lr to protect pretrained features
optimizer_tft = optim.Adam([
    {'params': alexnet_tft.features.parameters(), 'lr': 1e-5},    # backbone — gentle updates
    {'params': alexnet_tft.classifier.parameters(), 'lr': 1e-3}   # new head — learn fast
], weight_decay=5e-4)


print('=== Experiment C: True Fine-Tuning (pretrained, all layers unfrozen) ===')
alexnet_tft = train_model(
    model=alexnet_tft,
    optimizer=optimizer_tft,
    trainloader=cifar_trainloader,
    num_epochs=NUM_EPOCHS,
    run_name='alexnet_true_finetuning'
)

results['Experiment C (AlexNet True Fine-Tuning)'], _, _ = evaluate_model(
    alexnet_tft, cifar_testloader, 'AlexNet True Fine-Tuning (Pretrained)'
)
wandb.log({"test_accuracy": results['Experiment C (AlexNet True Fine-Tuning)']})
log_prediction_samples(alexnet_tft, cifar_testloader, 'exp_c_alexnet_tft')

end_run()

Parameters: 57,044,810 trainable / 57,044,810 total
=== Experiment C: True Fine-Tuning (pretrained, all layers unfrozen) ===
Epoch [1/10] | Loss: 0.7566 | Train Acc: 74.04%
Epoch [2/10] | Loss: 0.5797 | Train Acc: 80.21%
Epoch [3/10] | Loss: 0.5301 | Train Acc: 81.98%
Epoch [4/10] | Loss: 0.4995 | Train Acc: 82.77%
Epoch [5/10] | Loss: 0.4803 | Train Acc: 83.67%
Epoch [6/10] | Loss: 0.4592 | Train Acc: 84.29%
Epoch [7/10] | Loss: 0.4430 | Train Acc: 84.87%
Epoch [8/10] | Loss: 0.4307 | Train Acc: 85.31%
Epoch [9/10] | Loss: 0.4184 | Train Acc: 85.81%
Epoch [10/10] | Loss: 0.4087 | Train Acc: 86.02%
Training complete!
Test Accuracy (AlexNet True Fine-Tuning (Pretrained)): 86.42%


epoch,▁▂▃▃▄▅▆▆▇█
learning_rate_group_0,▁▁▁▁▁▁▁▁▁▁
learning_rate_group_1,▁▁▁▁▁▁▁▁▁▁
test_accuracy,▁
total_params,▁
trainable_params,▁
training_accuracy,▁▅▆▆▇▇▇███
training_loss,█▄▃▃▂▂▂▁▁▁
epoch,10
learning_rate_group_0,1e-05
learning_rate_group_1,0.001


CPU times: user 5min 40s, sys: 2min 11s, total: 7min 51s
Wall time: 11min 28s


### Why is there a difference between Experiments A, B and C?

**Experiment A (From Scratch — 80.65%)**  
The model starts with randomly initialized weights and must learn all useful features directly from CIFAR‑10. With only 10 epochs and a relatively large architecture (AlexNet), the network does not fully converge and cannot exploit its full capacity, so it underperforms compared to a model initialized with pretrained ImageNet weights. It still reaches reasonable accuracy, but needs more data and/or epochs to close the gap.

**Experiment B (Feature Extraction — 81.76%)**  
The convolutional layers already contain rich feature detectors learned from 1.2M ImageNet images (edges, textures, colors, basic shapes, object parts). These low‑ and mid‑level features transfer reasonably well to CIFAR‑10, even though the datasets are not identical. Only the final classification layer is trained, so the model converges quickly with fewer trainable parameters. However, because the backbone is frozen, it cannot adapt its features to CIFAR‑10‑specific statistics, so performance plateaus at a level similar to training from scratch.

**Experiment C (True Fine‑Tuning — 86.42%)**  
This setting combines the strengths of both approaches. The model starts from pretrained ImageNet features, giving it a strong initialization, and then all layers are unfrozen and trained with differential learning rates (a small learning rate for the backbone, a larger one for the classifier head). This allows the network to retain useful generic features while subtly reshaping them for CIFAR‑10. As a result, it converges quickly and achieves substantially higher test accuracy (about +5–6 percentage points over A and B).

**Analysis**  
Pretraining alone (B) already shows that generic ImageNet features are strong enough to match a from‑scratch AlexNet (A) in far fewer epochs. The largest gain, however, comes from **combining** pretraining with **end‑to‑end fine‑tuning** (C), which lets the model adapt its entire feature hierarchy to the new dataset instead of only retraining the final layer.

---
## Task 0.2.2 — Transfer Learning: MNIST → SVHN

### Load MNIST and SVHN datasets

In [98]:
!pip install scipy
!pip install scikit-learn

In [99]:
%%time

# MNIST: grayscale 28x28 images, 10 digit classes
# We resize to 32x32 and convert to 3 channels so the same CNN works on both datasets
mnist_transform = transforms.Compose([
    transforms.Resize(32),
    transforms.Grayscale(num_output_channels=3),  # Convert 1-channel to 3-channel
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# SVHN: color 32x32 images of house number digits, 10 classes
svhn_transform = transforms.Compose([
    transforms.Resize(32),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# MNIST
mnist_trainset = torchvision.datasets.MNIST(
    root='./data', train=True, download=True, transform=mnist_transform
)
mnist_testset = torchvision.datasets.MNIST(
    root='./data', train=False, download=True, transform=mnist_transform
)
mnist_trainloader = torch.utils.data.DataLoader(
    mnist_trainset, batch_size=64, shuffle=True, num_workers=2
)
mnist_testloader = torch.utils.data.DataLoader(
    mnist_testset, batch_size=64, shuffle=False, num_workers=2
)

# SVHN
svhn_trainset = torchvision.datasets.SVHN(
    root='./data', split='train', download=True, transform=svhn_transform
)
svhn_testset = torchvision.datasets.SVHN(
    root='./data', split='test', download=True, transform=svhn_transform
)
svhn_trainloader = torch.utils.data.DataLoader(
    svhn_trainset, batch_size=64, shuffle=True, num_workers=2
)
svhn_testloader = torch.utils.data.DataLoader(
    svhn_testset, batch_size=64, shuffle=False, num_workers=2
)

print('MNIST Training samples:', len(mnist_trainset))
print('MNIST Test samples:', len(mnist_testset))
print('SVHN Training samples:', len(svhn_trainset))
print('SVHN Test samples:', len(svhn_testset))

MNIST Training samples: 60000
MNIST Test samples: 10000
SVHN Training samples: 73257
SVHN Test samples: 26032
CPU times: user 3.82 s, sys: 535 ms, total: 4.35 s
Wall time: 3.99 s


### Define CNN (same architecture as labmate's Task 0.1)

In [100]:
%%time

class CNN(nn.Module):
    def __init__(self, activation_fn=nn.LeakyReLU()):
        super(CNN, self).__init__()
        self.activation_fn = activation_fn

        # Convolutional layers
        self.conv_block = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            self.activation_fn,
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            self.activation_fn,
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            self.activation_fn,
            nn.MaxPool2d(2, 2)
        )

        # Fully connected layers
        self.fc_block = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            self.activation_fn,
            nn.Dropout(p=0.5),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv_block(x)
        x = self.fc_block(x)
        return x

CPU times: user 223 μs, sys: 87 μs, total: 310 μs
Wall time: 314 μs


### Step 1 — Train CNN on MNIST

In [101]:
%%time

start_run('cnn_mnist', config={
    "architecture": "CNN",
    "dataset": "MNIST",
    "activation": "LeakyReLU",
    "optimizer": "Adam",
    "learning_rate": 0.001,
    "epochs": NUM_EPOCHS
})

cnn_mnist = CNN(activation_fn=nn.LeakyReLU()).to(device)
count_params(cnn_mnist)

optimizer_mnist = optim.Adam(cnn_mnist.parameters(), lr=1e-3, weight_decay=1e-4)

print('=== Training CNN on MNIST ===')
cnn_mnist = train_model(
    model=cnn_mnist,
    optimizer=optimizer_mnist,
    trainloader=mnist_trainloader,
    num_epochs=NUM_EPOCHS,
    run_name='cnn_mnist'
)

results['MNIST CNN'], _, _ = evaluate_model(cnn_mnist, mnist_testloader, 'CNN on MNIST')
wandb.log({"test_accuracy": results['MNIST CNN']})
log_prediction_samples(cnn_mnist, mnist_testloader, 'step1_mnist')

end_run()

Parameters: 620,362 trainable / 620,362 total
=== Training CNN on MNIST ===
Epoch [1/10] | Loss: 0.1748 | Train Acc: 94.46%
Epoch [2/10] | Loss: 0.0508 | Train Acc: 98.53%
Epoch [3/10] | Loss: 0.0389 | Train Acc: 98.84%
Epoch [4/10] | Loss: 0.0330 | Train Acc: 98.99%
Epoch [5/10] | Loss: 0.0286 | Train Acc: 99.10%
Epoch [6/10] | Loss: 0.0253 | Train Acc: 99.20%
Epoch [7/10] | Loss: 0.0217 | Train Acc: 99.38%
Epoch [8/10] | Loss: 0.0221 | Train Acc: 99.27%
Epoch [9/10] | Loss: 0.0180 | Train Acc: 99.44%
Epoch [10/10] | Loss: 0.0190 | Train Acc: 99.41%
Training complete!
Test Accuracy (CNN on MNIST): 99.09%


epoch,▁▂▃▃▄▅▆▆▇█
learning_rate_group_0,▁▁▁▁▁▁▁▁▁▁
test_accuracy,▁
total_params,▁
trainable_params,▁
training_accuracy,▁▇▇▇██████
training_loss,█▂▂▂▁▁▁▁▁▁
epoch,10
learning_rate_group_0,0.001
test_accuracy,99.09
total_params,620362


CPU times: user 1min 1s, sys: 8.31 s, total: 1min 9s
Wall time: 2min 53s


### Step 2 — Transfer the MNIST CNN to SVHN (Feature Extraction)

Freeze the convolutional layers (they've already learned useful features), keep the FC layers trainable, and fine-tune on SVHN.

In [102]:
%%time

start_run('cnn_mnist_to_svhn', config={
    "architecture": "CNN",
    "mode": "transfer_learning",
    "source_dataset": "MNIST",
    "target_dataset": "SVHN",
    "frozen_layers": "conv_block",
    "trainable_layers": "fc_block",
    "optimizer": "Adam",
    "learning_rate": 0.001,
    "epochs": NUM_EPOCHS
})

# Start from the MNIST-trained CNN
cnn_svhn = CNN(activation_fn=nn.LeakyReLU()).to(device)

# Copy the MNIST weights into the SVHN model
cnn_svhn.load_state_dict(cnn_mnist.state_dict())

# Freeze the conv_block (pretrained on MNIST)
for param in cnn_svhn.conv_block.parameters():
    param.requires_grad = False

count_params(cnn_svhn)

# Only train the fc_block on SVHN
optimizer_svhn = optim.Adam(
    filter(lambda p: p.requires_grad, cnn_svhn.parameters()),
    lr=1e-3,
    weight_decay=1e-4
)

print('=== Transfer Learning: MNIST CNN → SVHN ===')
cnn_svhn = train_model(
    model=cnn_svhn,
    optimizer=optimizer_svhn,
    trainloader=svhn_trainloader,
    num_epochs=NUM_EPOCHS,
    run_name='cnn_mnist_to_svhn'
)

results['SVHN (Transfer from MNIST)'], _, _ = evaluate_model(cnn_svhn, svhn_testloader, 'CNN Transfer MNIST→SVHN')
wandb.log({"test_accuracy": results['SVHN (Transfer from MNIST)']})
log_prediction_samples(cnn_svhn, svhn_testloader, 'step2_mnist_to_svhn_feature_extraction')

end_run()

Parameters: 527,114 trainable / 620,362 total
=== Transfer Learning: MNIST CNN → SVHN ===
Epoch [1/10] | Loss: 1.1236 | Train Acc: 64.48%
Epoch [2/10] | Loss: 0.8757 | Train Acc: 72.13%
Epoch [3/10] | Loss: 0.8079 | Train Acc: 74.18%
Epoch [4/10] | Loss: 0.7665 | Train Acc: 75.33%
Epoch [5/10] | Loss: 0.7437 | Train Acc: 75.92%
Epoch [6/10] | Loss: 0.7182 | Train Acc: 76.73%
Epoch [7/10] | Loss: 0.6999 | Train Acc: 77.33%
Epoch [8/10] | Loss: 0.6846 | Train Acc: 77.70%
Epoch [9/10] | Loss: 0.6728 | Train Acc: 78.08%
Epoch [10/10] | Loss: 0.6595 | Train Acc: 78.40%
Training complete!
Test Accuracy (CNN Transfer MNIST→SVHN): 77.72%


epoch,▁▂▃▃▄▅▆▆▇█
learning_rate_group_0,▁▁▁▁▁▁▁▁▁▁
test_accuracy,▁
total_params,▁
trainable_params,▁
training_accuracy,▁▅▆▆▇▇▇███
training_loss,█▄▃▃▂▂▂▁▁▁
epoch,10
learning_rate_group_0,0.001
test_accuracy,77.71589
total_params,620362


CPU times: user 57.5 s, sys: 10.7 s, total: 1min 8s
Wall time: 2min 41s


### Step 3 — Train the same CNN on SVHN from scratch (No Transfer Learning)

To verify whether the MNIST transfer actually helped, we train the identical CNN architecture directly on SVHN from random initialization.

In [103]:
%%time

start_run('cnn_svhn_scratch', config={
    "architecture": "CNN",
    "mode": "from_scratch",
    "dataset": "SVHN",
    "activation": "LeakyReLU",
    "optimizer": "Adam",
    "learning_rate": 0.001,
    "epochs": NUM_EPOCHS
})

cnn_svhn_scratch = CNN(activation_fn=nn.LeakyReLU()).to(device)
count_params(cnn_svhn_scratch)

optimizer_scratch = optim.Adam(cnn_svhn_scratch.parameters(), lr=2e-3, weight_decay=1e-4)

print('=== Baseline: CNN on SVHN from Scratch ===')
cnn_svhn_scratch = train_model(
    model=cnn_svhn_scratch,
    optimizer=optimizer_scratch,
    trainloader=svhn_trainloader,
    num_epochs=NUM_EPOCHS,
    run_name='cnn_svhn_scratch'
)

results['SVHN (From Scratch)'], _, _ = evaluate_model(cnn_svhn_scratch, svhn_testloader, 'CNN on SVHN from Scratch')
wandb.log({"test_accuracy": results['SVHN (From Scratch)']})
log_prediction_samples(cnn_svhn_scratch, svhn_testloader, 'step3_svhn_scratch')

end_run()

Parameters: 620,362 trainable / 620,362 total
=== Baseline: CNN on SVHN from Scratch ===
Epoch [1/10] | Loss: 0.9136 | Train Acc: 69.77%
Epoch [2/10] | Loss: 0.4282 | Train Acc: 87.10%
Epoch [3/10] | Loss: 0.3758 | Train Acc: 88.64%
Epoch [4/10] | Loss: 0.3454 | Train Acc: 89.63%
Epoch [5/10] | Loss: 0.3206 | Train Acc: 90.40%
Epoch [6/10] | Loss: 0.3009 | Train Acc: 90.89%
Epoch [7/10] | Loss: 0.2903 | Train Acc: 91.28%
Epoch [8/10] | Loss: 0.2807 | Train Acc: 91.59%
Epoch [9/10] | Loss: 0.2690 | Train Acc: 91.87%
Epoch [10/10] | Loss: 0.2594 | Train Acc: 92.18%
Training complete!
Test Accuracy (CNN on SVHN from Scratch): 90.67%


epoch,▁▂▃▃▄▅▆▆▇█
learning_rate_group_0,▁▁▁▁▁▁▁▁▁▁
test_accuracy,▁
total_params,▁
trainable_params,▁
training_accuracy,▁▆▇▇▇█████
training_loss,█▃▂▂▂▁▁▁▁▁
epoch,10
learning_rate_group_0,0.002
test_accuracy,90.67302
total_params,620362


CPU times: user 1min 9s, sys: 10.1 s, total: 1min 19s
Wall time: 2min 30s


### Step 4 — Transfer to SVHN with all layers unfrozen (Fine-Tuning)

Instead of freezing the conv layers, we unfreeze all layers and fine-tune with a smaller learning rate to gently adapt the MNIST features to SVHN. Since the domain gap between MNIST and SVHN is large (clean grayscale digits → cluttered color images), we thought the MNIST conv features likely need further adaptation even though both of them classify the digits.

In [104]:
%%time

start_run('cnn_mnist_to_svhn_finetune', config={
    "architecture": "CNN",
    "mode": "fine_tuning",
    "source_dataset": "MNIST",
    "target_dataset": "SVHN",
    "frozen_layers": "none",
    "optimizer": "Adam",
    "learning_rate": 0.0001,
    "epochs": NUM_EPOCHS
})

cnn_svhn_finetune = CNN(activation_fn=nn.LeakyReLU()).to(device)
cnn_svhn_finetune.load_state_dict(cnn_mnist.state_dict())
count_params(cnn_svhn_finetune)

# No freezing — all layers trainable, but use a smaller lr
optimizer_finetune = optim.Adam(cnn_svhn_finetune.parameters(), lr=1e-4, weight_decay=1e-4)

print('=== Fine-Tuning: MNIST CNN → SVHN (all layers unfrozen) ===')
cnn_svhn_finetune = train_model(
    model=cnn_svhn_finetune,
    optimizer=optimizer_finetune,
    trainloader=svhn_trainloader,
    num_epochs=NUM_EPOCHS,
    run_name='cnn_mnist_to_svhn_finetune'
)

results['SVHN (Fine-Tuned from MNIST)'], _, _ = evaluate_model(cnn_svhn_finetune, svhn_testloader, 'CNN Fine-Tuned MNIST→SVHN')
wandb.log({"test_accuracy": results['SVHN (Fine-Tuned from MNIST)']})
log_prediction_samples(cnn_svhn_finetune, svhn_testloader, 'step4_mnist_to_svhn_finetune')

end_run()

Parameters: 620,362 trainable / 620,362 total
=== Fine-Tuning: MNIST CNN → SVHN (all layers unfrozen) ===
Epoch [1/10] | Loss: 0.9619 | Train Acc: 69.87%
Epoch [2/10] | Loss: 0.5363 | Train Acc: 83.89%
Epoch [3/10] | Loss: 0.4367 | Train Acc: 86.93%
Epoch [4/10] | Loss: 0.3848 | Train Acc: 88.56%
Epoch [5/10] | Loss: 0.3457 | Train Acc: 89.79%
Epoch [6/10] | Loss: 0.3198 | Train Acc: 90.65%
Epoch [7/10] | Loss: 0.2953 | Train Acc: 91.26%
Epoch [8/10] | Loss: 0.2756 | Train Acc: 91.97%
Epoch [9/10] | Loss: 0.2580 | Train Acc: 92.50%
Epoch [10/10] | Loss: 0.2435 | Train Acc: 92.90%
Training complete!
Test Accuracy (CNN Fine-Tuned MNIST→SVHN): 91.62%


epoch,▁▂▃▃▄▅▆▆▇█
learning_rate_group_0,▁▁▁▁▁▁▁▁▁▁
test_accuracy,▁
total_params,▁
trainable_params,▁
training_accuracy,▁▅▆▇▇▇████
training_loss,█▄▃▂▂▂▂▁▁▁
epoch,10
learning_rate_group_0,0.0001
test_accuracy,91.61801
total_params,620362


CPU times: user 1min 9s, sys: 10.1 s, total: 1min 20s
Wall time: 2min 35s


### Why is there a difference between the four MNIST → SVHN experiments?

**Step 1 (CNN on MNIST — 99.09%)**  
The CNN is trained from scratch on MNIST and quickly reaches very high accuracy because MNIST is a relatively simple dataset: grayscale digits, clean backgrounds, and centered objects. With 10 epochs and a reasonably expressive architecture, the network can almost perfectly fit the digit patterns, leading to ≈99% test accuracy and only a small train–test gap.

**Step 2 (Transfer MNIST → SVHN, Feature Extraction — 77.72%)**  
Here the convolutional layers pretrained on MNIST are frozen and only the fully connected layers are trained on SVHN. The problem is that MNIST and SVHN differ significantly: MNIST is grayscale and clean, while SVHN has colored digits, varying positions, cluttered backgrounds, and more complex visual statistics. The frozen MNIST features therefore do not align well with SVHN, so the network cannot learn good representations for the new domain. As a result, both train and test accuracy saturate around 77–78%, far below what is achievable when training directly on SVHN.

**Step 3 (SVHN from Scratch — 90.67%)**  
The same CNN is now trained from scratch on SVHN, without any MNIST pretraining. Even though learning starts from random weights, the model is free to adapt all layers to the SVHN distribution. With 10 epochs, it learns strong domain‑appropriate features (colors, backgrounds, digit styles) and achieves ≈90.7% test accuracy. This clearly outperforms the frozen‑feature transfer from MNIST, showing that a mismatched source domain can be worse than no pretraining at all when the backbone is not allowed to change.

**Step 4 (MNIST → SVHN Fine‑Tuning — 91.62%)**  
In this setting, the CNN is initialized with MNIST‑trained weights, but **all** layers are unfrozen and trained on SVHN with a smaller learning rate. The initial MNIST features give the network a starting point for digit‑like structures, while full fine‑tuning allows it to gradually adapt to SVHN’s colors, backgrounds, and style. This combination slightly improves over the SVHN‑from‑scratch baseline, reaching ≈91.6% test accuracy, and demonstrates that pretraining can still be beneficial when the entire network is allowed to adapt to the new domain.

**Analysis**  
These four steps illustrate how **domain similarity and flexibility of adaptation** determine transfer learning success. Pure feature extraction from a very different source (MNIST → SVHN with frozen conv layers) leads to weak or negative transfer, performing much worse than a model trained from scratch on SVHN. When the model can **fine‑tune all layers**, MNIST pretraining becomes a helpful initialization rather than a constraint, yielding a modest but real improvement over the SVHN scratch baseline. This contrasts with the ImageNet → CIFAR‑10 case, where the domains are closer and even frozen feature extraction is already competitive, highlighting that the effectiveness of transfer learning is highly domain‑dependent.

---
## Final Results Summary

In [105]:
print(f"{'Experiment':<45} {'Test Accuracy':>15}")
print('-' * 62)

# Loop with matching alignment
for name, acc in results.items():
    print(f"{name:<45} {acc:>14.2f}%")

Experiment                                      Test Accuracy
--------------------------------------------------------------
Experiment A (AlexNet Fine-Tuning)                     80.65%
Experiment B (AlexNet Feature Extraction)              81.76%
Experiment C (AlexNet True Fine-Tuning)                86.42%
MNIST CNN                                              99.09%
SVHN (Transfer from MNIST)                             77.72%
SVHN (From Scratch)                                    90.67%
SVHN (Fine-Tuned from MNIST)                           91.62%


## Visualisations

Per-class accuracy reports and interactive confusion matrix heatmaps for all models across both tasks. These help identify which classes each approach handles well, where models struggle, and how transfer learning shifts the error patterns.


### CIFAR-10 (AlexNet — Experiments A, B, C)

CIFAR-10 has semantically similar class pairs (cat/dog, car/truck, deer/horse) that make confusion patterns especially informative.

In [106]:
cifar_classes = ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

cifar_models = {
    'CIFAR-10 (From Scratch)':        alexnet_ft,
    'CIFAR-10 (Feature Extraction)':  alexnet_fe,
    'CIFAR-10 (True Fine-Tuning)':    alexnet_tft,
}

for name, model in cifar_models.items():
    _, preds, labels = evaluate_model(model, cifar_testloader, name)
    print(f'\n=== Per-Class Report: {name} ===')
    print(classification_report(labels, preds, target_names=cifar_classes))

    # Log interactive confusion matrix heatmap to WandB
    wandb_cm = wandb.plot.confusion_matrix(
        y_true=labels, preds=preds,
        class_names=cifar_classes,
        title=f'Confusion Matrix: {name}'
    )
    start_run(f'analysis_{name.replace(" ", "_").replace("(", "").replace(")", "").lower()}', config={"type": "analysis"})
    wandb.log({"confusion_matrix": wandb_cm})
    end_run()
    print('=' * 70)

Test Accuracy (CIFAR-10 (From Scratch)): 80.65%

=== Per-Class Report: CIFAR-10 (From Scratch) ===
              precision    recall  f1-score   support

       plane       0.78      0.85      0.81      1000
         car       0.91      0.88      0.90      1000
        bird       0.86      0.62      0.72      1000
         cat       0.67      0.66      0.66      1000
        deer       0.72      0.88      0.79      1000
         dog       0.76      0.74      0.75      1000
        frog       0.91      0.78      0.84      1000
       horse       0.87      0.86      0.86      1000
        ship       0.92      0.85      0.89      1000
       truck       0.75      0.95      0.84      1000

    accuracy                           0.81     10000
   macro avg       0.81      0.81      0.81     10000
weighted avg       0.81      0.81      0.81     10000



Test Accuracy (CIFAR-10 (Feature Extraction)): 81.76%

=== Per-Class Report: CIFAR-10 (Feature Extraction) ===
              precision    recall  f1-score   support

       plane       0.83      0.83      0.83      1000
         car       0.82      0.94      0.88      1000
        bird       0.84      0.74      0.78      1000
         cat       0.73      0.64      0.68      1000
        deer       0.73      0.83      0.78      1000
         dog       0.81      0.76      0.78      1000
        frog       0.85      0.86      0.85      1000
       horse       0.83      0.85      0.84      1000
        ship       0.86      0.89      0.87      1000
       truck       0.89      0.83      0.86      1000

    accuracy                           0.82     10000
   macro avg       0.82      0.82      0.82     10000
weighted avg       0.82      0.82      0.82     10000



Test Accuracy (CIFAR-10 (True Fine-Tuning)): 86.42%

=== Per-Class Report: CIFAR-10 (True Fine-Tuning) ===
              precision    recall  f1-score   support

       plane       0.89      0.86      0.88      1000
         car       0.94      0.85      0.89      1000
        bird       0.94      0.77      0.85      1000
         cat       0.88      0.61      0.72      1000
        deer       0.84      0.92      0.88      1000
         dog       0.81      0.85      0.83      1000
        frog       0.83      0.96      0.89      1000
       horse       0.91      0.91      0.91      1000
        ship       0.92      0.92      0.92      1000
       truck       0.76      0.98      0.85      1000

    accuracy                           0.86     10000
   macro avg       0.87      0.86      0.86     10000
weighted avg       0.87      0.86      0.86     10000



### SVHN (MNIST→SVHN — Steps 1, 2, 3, 4)

Both MNIST and SVHN classify digits 0–9, but SVHN's cluttered real-world backgrounds and varying fonts make certain digit pairs harder to distinguish (e.g., 3/8, 1/7, 5/6).

In [107]:
mnist_svhn_models = {
    'MNIST CNN':                    (cnn_mnist,        mnist_testloader),
    'SVHN (Transfer — Frozen Conv)':(cnn_svhn,         svhn_testloader),
    'SVHN (From Scratch)':          (cnn_svhn_scratch, svhn_testloader),
    'SVHN (Fine-Tuned from MNIST)': (cnn_svhn_finetune,svhn_testloader),
}

for name, (model, loader) in mnist_svhn_models.items():
    _, preds, labels = evaluate_model(model, loader, name)
    print(f'\n=== Per-Class Report: {name} ===')
    print(classification_report(labels, preds, target_names=[str(i) for i in range(10)]))
    
    # Log interactive confusion matrix heatmap to WandB
    wandb_cm = wandb.plot.confusion_matrix(
        y_true=labels, preds=preds,
        class_names=[str(i) for i in range(10)],
        title=f'Confusion Matrix: {name}'
    )
    start_run(f'analysis_{name.replace(" ", "_").replace("—", "").lower()}', config={"type": "analysis"})
    wandb.log({"confusion_matrix": wandb_cm})
    end_run()
    print('=' * 70)


Test Accuracy (MNIST CNN): 99.09%

=== Per-Class Report: MNIST CNN ===
              precision    recall  f1-score   support

           0       0.99      1.00      0.99       980
           1       0.99      0.99      0.99      1135
           2       0.99      0.99      0.99      1032
           3       0.98      1.00      0.99      1010
           4       0.99      0.99      0.99       982
           5       0.99      0.99      0.99       892
           6       1.00      0.99      0.99       958
           7       0.98      0.99      0.99      1028
           8       1.00      0.98      0.99       974
           9       0.99      0.99      0.99      1009

    accuracy                           0.99     10000
   macro avg       0.99      0.99      0.99     10000
weighted avg       0.99      0.99      0.99     10000



Test Accuracy (SVHN (Transfer — Frozen Conv)): 77.72%

=== Per-Class Report: SVHN (Transfer — Frozen Conv) ===
              precision    recall  f1-score   support

           0       0.79      0.73      0.76      1744
           1       0.79      0.89      0.84      5099
           2       0.81      0.82      0.82      4149
           3       0.75      0.70      0.73      2882
           4       0.76      0.83      0.79      2523
           5       0.74      0.78      0.76      2384
           6       0.75      0.70      0.72      1977
           7       0.84      0.78      0.81      2019
           8       0.75      0.62      0.68      1660
           9       0.73      0.67      0.70      1595

    accuracy                           0.78     26032
   macro avg       0.77      0.75      0.76     26032
weighted avg       0.78      0.78      0.78     26032



Test Accuracy (SVHN (From Scratch)): 90.67%

=== Per-Class Report: SVHN (From Scratch) ===
              precision    recall  f1-score   support

           0       0.95      0.86      0.90      1744
           1       0.93      0.96      0.94      5099
           2       0.96      0.92      0.94      4149
           3       0.90      0.87      0.88      2882
           4       0.94      0.92      0.93      2523
           5       0.85      0.93      0.89      2384
           6       0.78      0.92      0.85      1977
           7       0.92      0.92      0.92      2019
           8       0.94      0.76      0.84      1660
           9       0.85      0.89      0.87      1595

    accuracy                           0.91     26032
   macro avg       0.90      0.89      0.90     26032
weighted avg       0.91      0.91      0.91     26032



Test Accuracy (SVHN (Fine-Tuned from MNIST)): 91.62%

=== Per-Class Report: SVHN (Fine-Tuned from MNIST) ===
              precision    recall  f1-score   support

           0       0.91      0.90      0.90      1744
           1       0.94      0.95      0.95      5099
           2       0.93      0.95      0.94      4149
           3       0.89      0.89      0.89      2882
           4       0.95      0.92      0.93      2523
           5       0.94      0.88      0.91      2384
           6       0.87      0.90      0.88      1977
           7       0.93      0.91      0.92      2019
           8       0.88      0.87      0.87      1660
           9       0.85      0.89      0.87      1595

    accuracy                           0.92     26032
   macro avg       0.91      0.91      0.91     26032
weighted avg       0.92      0.92      0.92     26032

